# Clustering Based Disease Diagnosis
## Laboratory for Numerical Medical Data

### Aim
To apply unsupervised machine learning techniques on numerical medical data and identify groups of patients with similar clinical characteristics.

### Objectives
- Perform exploratory data analysis on numerical medical data.
- Standardize clinical features.
- Apply Hierarchical Agglomerative Clustering.
- Construct dendrograms using different linkage methods.
- Implement K-Means and K-Medoids clustering.
- Evaluate clusters using Silhouette Score and Inertia.
- Visualize clusters using Principal Component Analysis (PCA).
- Interpret clusters based on clinical measurements.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import silhouette_score, pairwise_distances
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings("ignore")

## 1. Dataset Creation

The dataset contains numerical clinical measurements of 12 fictional patients.

Features:
- Glucose
- BMI
- Systolic Blood Pressure
- HbA1c

No diagnosis or target label is provided because this experiment uses unsupervised learning.

In [ ]:
data = {
    "ID": ["P01", "P02", "P03", "P04", "P05", "P06",
           "P07", "P08", "P09", "P10", "P11", "P12"],

    "Glucose": [92, 105, 110, 125, 138, 148, 160, 98, 118, 132, 88, 155],

    "BMI": [22.4, 24.1, 26.5, 28.2, 30.4, 32.1, 34.5, 23.0, 27.1, 29.8, 21.7, 33.0],

    "Systolic_BP": [118, 122, 128, 132, 140, 146, 152, 120, 130, 136, 114, 150],

    "HbA1c": [5.1, 5.4, 5.8, 6.1, 6.7, 7.2, 7.8, 5.2, 6.0, 6.5, 4.9, 7.5]
}

df = pd.DataFrame(data)

df

## 2. Data Inspection

Inspect the dataset, data types, descriptive statistics, missing values, and duplicate observations.

In [ ]:
print("Dataset Shape:", df.shape)

print("\nData Types:")
print(df.dtypes)

print("\nDescriptive Statistics:")
print(df.describe())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

## 3. Feature Selection

Select the four numerical clinical features for clustering.

The patient ID is retained only for identification and is not used as a clustering feature.

In [ ]:
features = ["Glucose", "BMI", "Systolic_BP", "HbA1c"]

X = df[features]

print(X)

: 

## 4. Standardization

Standardize the numerical features using StandardScaler.

Standardization transforms the features to approximately zero mean and unit variance.

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(
    X_scaled,
    columns=features,
    index=df["ID"]
)

print(X_scaled_df.round(2))

## 5. Hierarchical Agglomerative Clustering

Hierarchical Agglomerative Clustering is a bottom-up clustering algorithm.

Different linkage methods are used to construct dendrograms:
- Ward Linkage
- Complete Linkage
- Average Linkage

In [ ]:
Z_ward = linkage(
    X_scaled,
    method="ward",
    metric="euclidean"
)

plt.figure(figsize=(12, 6))

dendrogram(
    Z_ward,
    labels=df["ID"].to_numpy(),
    leaf_rotation=45
)

plt.title("Ward Linkage Dendrogram")
plt.xlabel("Patient ID")
plt.ylabel("Merge Distance")

plt.tight_layout()
plt.show()

In [ ]:
Z_complete = linkage(
    X_scaled,
    method="complete",
    metric="euclidean"
)

plt.figure(figsize=(12, 6))

dendrogram(
    Z_complete,
    labels=df["ID"].to_numpy(),
    leaf_rotation=45
)

plt.title("Complete Linkage Dendrogram")
plt.xlabel("Patient ID")
plt.ylabel("Merge Distance")

plt.tight_layout()
plt.show()

In [ ]:
Z_average = linkage(
    X_scaled,
    method="average",
    metric="euclidean"
)

plt.figure(figsize=(12, 6))

dendrogram(
    Z_average,
    labels=df["ID"].to_numpy(),
    leaf_rotation=45
)

plt.title("Average Linkage Dendrogram")
plt.xlabel("Patient ID")
plt.ylabel("Merge Distance")

plt.tight_layout()
plt.show()

## 6. HAC Clustering with Three Clusters

Cut the Ward dendrogram to obtain three clusters and evaluate the clustering using the Silhouette Score.

In [ ]:
hac_labels = fcluster(
    Z_ward,
    t=3,
    criterion="maxclust"
) - 1

df["HAC_Cluster"] = hac_labels

print("HAC Cluster Assignments:")
print(df[["ID", "HAC_Cluster"]])

print("\nHAC Cluster Sizes:")
print(df["HAC_Cluster"].value_counts().sort_index())

hac_silhouette = silhouette_score(
    X_scaled,
    hac_labels
)

print("\nHAC Silhouette Score:")
print(round(hac_silhouette, 4))

## 7. K-Means Clustering

Apply K-Means clustering with three clusters using standardized clinical features.

In [ ]:
k = 3

kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=20
)

kmeans_labels = kmeans.fit_predict(X_scaled)

df["KMeans_Cluster"] = kmeans_labels

print("K-Means Cluster Assignments:")
print(df[["ID", "KMeans_Cluster"]])

print("\nCluster Sizes:")
print(df["KMeans_Cluster"].value_counts().sort_index())

print("\nCentroids:")
centroids = pd.DataFrame(
    kmeans.cluster_centers_,
    columns=features
)

print(centroids.round(2))

print("\nInertia:")
print(round(kmeans.inertia_, 4))

print("\nSilhouette Score:")
print(round(silhouette_score(X_scaled, kmeans_labels), 4))

## 8. K-Means for Different Values of K

Apply K-Means for k values from 2 to 5.

Calculate inertia and silhouette score for each value of k.

In [ ]:

inertias = []
silhouettes = []

k_values = range(2, 6)

for k_value in k_values:

    model = KMeans(
        n_clusters=k_value,
        random_state=42,
        n_init=20
    )

    labels = model.fit_predict(X_scaled)

    inertias.append(model.inertia_)

    silhouettes.append(
        silhouette_score(X_scaled, labels)
    )

results_kmeans = pd.DataFrame({
    "K": list(k_values),
    "Inertia": inertias,
    "Silhouette": silhouettes
})

print(results_kmeans.round(4))


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    list(k_values),
    inertias,
    marker="o"
)

plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.title("K-Means Elbow Plot")

plt.xticks(list(k_values))
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    list(k_values),
    silhouettes,
    marker="o"
)

plt.xlabel("Number of Clusters (k)")
plt.ylabel("Silhouette Score")
plt.title("K-Means Silhouette Analysis")

plt.xticks(list(k_values))
plt.grid(True)

plt.tight_layout()
plt.show()

## 9. K-Medoids Clustering

Implement K-Medoids using pairwise Euclidean distances.

A medoid is an actual observation that minimizes the total distance to other observations in its cluster.

In [ ]:
def kmedoids(X, n_clusters, random_state=42, max_iter=100):

    rng = np.random.default_rng(random_state)

    D = pairwise_distances(
        X,
        metric="euclidean"
    )

    medoids = rng.choice(
        len(X),
        size=n_clusters,
        replace=False
    )

    for iteration in range(max_iter):

        labels = np.argmin(
            D[:, medoids],
            axis=1
        )

        new_medoids = medoids.copy()

        for c in range(n_clusters):

            members = np.where(labels == c)[0]

            if len(members) == 0:
                continue

            costs = D[
                np.ix_(members, members)
            ].sum(axis=1)

            new_medoids[c] = members[
                np.argmin(costs)
            ]

        if np.array_equal(new_medoids, medoids):
            break

        medoids = new_medoids

    labels = np.argmin(
        D[:, medoids],
        axis=1
    )

    return labels, medoids

In [ ]:
medoid_labels, medoid_indices = kmedoids(
    X_scaled,
    n_clusters=3,
    random_state=42
)

df["KMedoids_Cluster"] = medoid_labels

print("K-Medoids Cluster Assignments:")
print(df[["ID", "KMedoids_Cluster"]])

print("\nCluster Sizes:")
print(df["KMedoids_Cluster"].value_counts().sort_index())

print("\nMedoid Patient IDs:")
print(df.iloc[medoid_indices]["ID"].tolist())

print("\nSilhouette Score:")
print(round(silhouette_score(X_scaled, medoid_labels), 4))

In [ ]:
kmeans_profiles = df.groupby(
    "KMeans_Cluster"
)[features].mean()

print("K-Means Cluster Profiles:")
print(kmeans_profiles.round(2))

In [ ]:
hac_profiles = df.groupby(
    "HAC_Cluster"
)[features].mean()

print("HAC Cluster Profiles:")
print(hac_profiles.round(2))

In [ ]:
medoids_profiles = df.groupby(
    "KMedoids_Cluster"
)[features].mean()

print("K-Medoids Cluster Profiles:")
print(medoids_profiles.round(2))

In [ ]:
print("HAC Cluster Sizes:")
print(df["HAC_Cluster"].value_counts().sort_index())

print("\nK-Means Cluster Sizes:")
print(df["KMeans_Cluster"].value_counts().sort_index())

print("\nK-Medoids Cluster Sizes:")
print(df["KMedoids_Cluster"].value_counts().sort_index())

## 11. Principal Component Analysis (PCA)

Apply PCA to reduce the four standardized clinical features into two principal components for visualization.

Clustering is performed using all four standardized features. PCA is used only for plotting.

In [ ]:
pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_scaled)

print("Explained Variance Ratio:")
print(pca.explained_variance_ratio_.round(4))

print("\nTotal Variance Explained:")
print(round(sum(pca.explained_variance_ratio_), 4))

In [ ]:
fig, axes = plt.subplots(
    1, 3,
    figsize=(16, 5)
)

labels_list = [
    hac_labels,
    kmeans_labels,
    medoid_labels
]

titles = [
    "HAC Clustering",
    "K-Means Clustering",
    "K-Medoids Clustering"
]

for ax, labels, title in zip(
    axes,
    labels_list,
    titles
):

    ax.scatter(
        X_pca[:, 0],
        X_pca[:, 1],
        c=labels,
        cmap="viridis",
        s=100
    )

    for i, patient_id in enumerate(df["ID"]):

        ax.annotate(
            patient_id,
            (X_pca[i, 0], X_pca[i, 1]),
            fontsize=8
        )

    ax.set_title(title)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

plt.tight_layout()
plt.show()

## 12. Performance Comparison

Compare HAC, K-Means, and K-Medoids using:

- Silhouette Score
- Inertia
- Cluster Sizes
- Representative Patients

In [ ]:
comparison = pd.DataFrame({
    "Method": [
        "HAC",
        "K-Means",
        "K-Medoids"
    ],

    "K": [
        3,
        3,
        3
    ],

    "Silhouette": [
        silhouette_score(X_scaled, hac_labels),
        silhouette_score(X_scaled, kmeans_labels),
        silhouette_score(X_scaled, medoid_labels)
    ],

    "Inertia": [
        np.nan,
        kmeans.inertia_,
        np.nan
    ],

    "Cluster Sizes": [
        df["HAC_Cluster"].value_counts().sort_index().tolist(),
        df["KMeans_Cluster"].value_counts().sort_index().tolist(),
        df["KMedoids_Cluster"].value_counts().sort_index().tolist()
    ],

    "Representative Patients": [
        "Not applicable",
        "Centroids",
        df.iloc[medoid_indices]["ID"].tolist()
    ]
})

print(comparison.round(4))

## 13. Final Cluster Assignment Table

Display the original clinical measurements along with cluster assignments generated by all three algorithms.

In [ ]:
final_df = df[
    [
        "ID",
        "Glucose",
        "BMI",
        "Systolic_BP",
        "HbA1c",
        "HAC_Cluster",
        "KMeans_Cluster",
        "KMedoids_Cluster"
    ]
]

final_df

## 14. Final Results

| Method | K | Cluster Sizes | Silhouette Score | Inertia | Representative Patients |
|---|---|---|---|---|---|
| HAC | 3 | | | N/A | N/A |
| K-Means | 3 | | | | Centroids |
| K-Medoids | 3 | | | N/A | |

### Clinical Interpretation

| Cluster | Glucose | BMI | Systolic BP | HbA1c | Interpretation |
|---|---|---|---|---|---|
| 0 | | | | | |
| 1 | | | | | |
| 2 | | | | | |

### Conclusion

Unsupervised clustering was performed on numerical medical data using Hierarchical Agglomerative Clustering, K-Means, and K-Medoids.

The features were standardized before clustering. Cluster quality was evaluated using silhouette scores, cluster sizes, inertia, dendrograms, and PCA visualizations.

The experiment identified groups of patients with similar numerical clinical characteristics. The clusters are exploratory and do not establish a medical diagnosis.